In [70]:
import torch
from einops import rearrange, einsum, reduce
from torch import nn

In [ ]:
# ALl neural nets modules should inherent from nn.Module parent class -> inherits convenient methods such as: load_state_dict(), to(), get_parameters(), cpu(), cuda(), children(), bfloat16()...

# Implement a Linear Class (= "a Linear Module")
# y = xWT

class Linear(nn.Module): # Inherits nn.Module methods()
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        
        self.sigma = (2 / (in_features + out_features)) ** (1/2)

        self.W = nn.Parameter(nn.init.trunc_normal_(torch.empty(out_features, in_features, dtype=dtype, device=device), 
                                                                mean=0, std = self.sigma, 
                                                                a = -3 * self.sigma, b= 3 * self.sigma ))

    def forward(self, x: torch.tensor) -> torch.Tensor: # All nn.Module need to have a forward() method
        return einsum(x, self.W, '... in_feature , out_feature in_feature -> ... out_feature')

In [72]:
import einops
ll = Linear(10,3)
x = torch.arange(1,11, dtype=torch.float32)
x_batched = einops.repeat(x, 'l -> b1 l', b1=4)
print(x_batched)
print(ll(x_batched))
print('---')
ll = Linear(3, 5)
weights = torch.ones(5, 3)
ll.load_state_dict({"W": weights})
print(ll.W)

tensor([[ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]])
tensor([[ 2.6985, 15.3634,  5.0632],
        [ 2.6985, 15.3634,  5.0632],
        [ 2.6985, 15.3634,  5.0632],
        [ 2.6985, 15.3634,  5.0632]], grad_fn=<ViewBackward0>)
---
Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)
